# Act 1 — The Architecture Framework, with services attached

The GCP Architecture Framework is six pillars: **Operational Excellence**, **Security/Privacy/Compliance**, **Reliability**, **Cost Optimization**, **Performance Optimization**, **Sustainability**. Notebook 01 introduced them as a vocabulary; this act maps each one to the concrete GCP services we've been building with.

## Pillar → service mapping

**Operational Excellence**

- Cloud Build + Cloud Deploy for repeatable releases.
- Cloud Logging + Monitoring + Trace + Profiler + Error Reporting for the observability stack.
- Terraform for infrastructure-as-code.
- Workload Identity Federation for keyless CI.
- SLOs and burn-rate alerts to align ops on user-visible reliability.

**Security, Privacy & Compliance**

- The IAM cascade (hierarchy + bindings + conditions) — notebook 02.
- Org Policy + Hierarchical Firewall + Deny policies — notebook 02.
- VPC Service Controls — notebook 06 by reference (covered as the GCP-unique exfil boundary).
- Cloud KMS + Secret Manager + Cloud HSM — notebook 11.
- Workload Identity (GKE) + WIF for external workloads — notebooks 02 and 11.
- Binary Authorization for image provenance — notebook 11.
- Security Command Center as the umbrella — notebook 11.
- Cloud Armor + IAP at the edge — notebooks 07 and 11.
- Cloud Audit Logs sinked to BigQuery — notebook 12.

**Reliability**

- Regional MIGs, regional GKE clusters, regional Cloud SQL HA — notebooks 03, 04, 08.
- Multi-region Cloud Storage and BigQuery for data resilience — notebooks 05 and 09.
- Spanner for globally-consistent writes — notebook 09.
- Disaster recovery strategy: Backup/Restore, Pilot Light, Warm Standby, Active-Active — notebook 13.
- Backup and DR Service for managed backups — notebook 13.

**Cost Optimization**

- Sustained Use Discounts (automatic) + Committed Use Discounts (spend-based or resource-based) + Spot VMs — notebook 03.
- BigQuery partitioning + clustering + Editions vs on-demand — notebook 09.
- GCS lifecycle rules (Standard → Nearline → Archive → Delete) — notebook 05.
- Cloud Run scale-to-zero for low-traffic workloads — notebook 04.
- Right-sizing via Active Assist (Recommender) — this notebook, next act.

**Performance Optimization**

- Global External Application LB + Cloud CDN + Cloud Armor for low-latency public traffic — notebook 07.
- Memorystore as cache layer in front of Cloud SQL — notebook 08.
- BI Engine for sub-second BigQuery dashboards — notebook 09.
- Direct VPC Egress on Cloud Run for faster internal calls — notebook 04.

**Sustainability**

- Carbon-free energy percentages per region (notebook 01) — pick `europe-north1`, `us-central1` for sustainability-sensitive workloads.
- Right-sizing reduces idle hardware, reducing emissions.
- Managed services (Cloud Run, Cloud Functions) amortise hardware across many tenants.

**The framework isn't a checklist.** It's a vocabulary for trade-offs. A workload that's perfectly cost-optimized may be under-secured; a workload that's perfectly reliable may be over-budget. The pillars are useful as the lens for asking "what did I trade off here?"

# Act 2 — Billing and cost levers

Cost optimization deserves its own act because the levers are concrete and the savings are often large. Three structural concepts first, then the GCP-specific cost levers.

## Billing structure

- **Billing account** — the entity invoiced. Linked to one or more projects. Can have multiple billing accounts per Organization (separate cost centres).
- **Billing hierarchy** — projects inherit charges to their billing account. Move a project between billing accounts to re-attribute spend.
- **Budgets** — per billing account or per project. Define threshold rules (50%, 90%, 100%); on hit, send Pub/Sub messages or email. Budget *alerts*, not budget *enforcement* — GCP won't shut down resources to stay under budget.
- **BigQuery billing export** — export detailed line items to BigQuery for SQL analysis. The single highest-leverage cost-tooling decision; turn this on the day you create the billing account.
- **Cost Insights and Recommender** — Active Assist's cost surface. Surfaces "this VM is idle, kill it" and "this commitment would save $X."

## The big cost levers, in order of impact

1. **Sustained Use Discounts** — already on. Verify by reading the SUD line on invoices.
2. **Committed Use Discounts** — for any steady-state baseline (Compute Engine, Cloud SQL, GKE). Start with spend-based CUDs at 1-year for safety; move to 3-year resource-based once usage stabilises.
3. **Spot VMs for fault-tolerant work** — batch, build agents, render farms. 60–91% off.
4. **BigQuery partitioning** — cuts on-demand query cost dramatically. The most common single optimisation for established BQ tables.
5. **GCS lifecycle rules** — Standard → Nearline → Archive → Delete. Saves more than any other storage optimisation.
6. **Right-size VMs** — Recommender surfaces over-provisioned instances. Apply its suggestions monthly.
7. **Idle resource cleanup** — orphaned PDs, unused IPs, forgotten BigQuery datasets. Cloud Asset Inventory + scheduled cleanup scripts catch these.
8. **Cloud Run min-instances audit** — `min=1+` on a low-traffic service is effectively a small Compute Engine VM, billed by the millisecond. Check whether you actually need it.

**The FinOps habit:** monthly cost-review meeting against the billing export. Bring the top 5 line items and ask what changed.

# Act 3 — Associate Cloud Engineer exam prep

The ACE blueprint covers most of what's in this course. Three things to do in the last 1–2 weeks before taking the exam: cover the blueprint, internalise the keyword-to-service playbook, and recognise the distractor traps.

## ACE blueprint coverage

The ACE blueprint (as of 2026) breaks roughly into:

| Domain | Weight | Mapped to |
|---|---|---|
| Setting up a cloud solution environment | ~17.5% | Notebook 01 + 02 + 14 (billing) |
| Planning and configuring a cloud solution | ~17.5% | Notebooks 03, 04, 05, 06 |
| Deploying and implementing a cloud solution | ~25% | Notebooks 03, 04, 05, 06, 07, 13 |
| Ensuring successful operation | ~20% | Notebook 12 + 13 |
| Configuring access and security | ~20% | Notebooks 02, 11 |

Nothing on the blueprint is missing from this course. If you've worked through the notebooks honestly, the gap to ACE-ready is mostly hands-on practice in the console — labs, `gcloud` muscle memory, and reading the Google docs for the specific products you don't yet have personal experience with.

## Keyword → service playbook

When you see one of these phrases in an exam question, the answer is usually the service in the right column.

| Keyword | Service |
|---|---|
| "Globally distributed SQL" / "strong consistency at scale" | Spanner |
| "Stream with replay" / "event-driven fan-out" | Pub/Sub |
| "Data warehouse" / "SQL over TBs" | BigQuery |
| "WAF at the edge" / "global DDoS protection" | Cloud Armor |
| "Federated identity for CI" / "keyless GitHub Actions" | Workload Identity Federation |
| "Zero-trust web access" / "no VPN, no bastion" | IAP |
| "Container image policy at deploy" | Binary Authorization |
| "Encryption key under customer control" | Cloud KMS (CMEK) |
| "Application secret storage" | Secret Manager |
| "Single VPC across regions" | (it's already that — GCP VPC is global) |
| "Hybrid connectivity, low bandwidth" | HA VPN |
| "Hybrid connectivity, high bandwidth" | Dedicated / Partner Interconnect |
| "Connect to SaaS / managed service privately" | Private Service Connect |
| "Synchronous storage replication" | Dual-region Cloud Storage / Regional PD |
| "Cron on GCP" | Cloud Scheduler |
| "Multi-step orchestration with API calls" | Workflows |
| "Rate-controlled HTTP fan-out" | Cloud Tasks |
| "Real-time mobile app database" | Firestore |
| "Petabyte time-series" | Bigtable |
| "Lift-and-shift database" | Database Migration Service |
| "Multi-cloud analytics without copying data" | BigQuery Omni |

## Distractor traps

Question writers love services that *look* like the answer but aren't. Watch for these:

- **Spanner vs Cloud SQL vs AlloyDB vs BigQuery.** Spanner is for global transactional writes (not analytics). BigQuery is for analytics (not OLTP). AlloyDB is Postgres-shaped OLTP at scale. Cloud SQL is the default.
- **Pub/Sub vs Eventarc vs Workflows vs Cloud Tasks.** Pub/Sub for fan-out. Eventarc for Google-source events. Workflows for multi-step orchestration. Cloud Tasks for rate-controlled HTTP.
- **Cloud Run vs Cloud Functions vs GKE Autopilot.** Cloud Functions for one event-handler. Cloud Run for HTTP services. GKE Autopilot when you genuinely need Kubernetes shape.
- **Cloud Armor edge security policy vs security policy.** Edge runs before Cloud CDN; security policy runs at the backend. Don't confuse them.
- **VPC Peering vs Shared VPC vs PSC.** Peering = two VPCs see each other. Shared VPC = one VPC, multiple projects. PSC = one consumer reaches one producer privately.
- **Org Policy vs IAM vs Deny policy.** Org Policy = what can exist. IAM = who can act. Deny policy = explicit override.
- **Workload Identity vs Workload Identity Federation.** WI = GKE K8s SA → Google SA. WIF = external (GitHub etc) → Google SA. Both impersonate; the source is different.
- **GCS class vs location.** Independent axes. "Nearline in us-central1" and "Nearline in us" are different things.
- **Standard vs Premium network tier.** Premium uses Google backbone end-to-end. Standard rides public internet to the destination region.
- **Cloud SQL HA failover vs cross-region read replica promotion.** HA failover is automatic, in-region, seconds. Cross-region promotion is manual, for DR.

## Study plan — the last two weeks

1. **Week 1 — re-read this course's notebooks 02, 06, 07, 11.** These are the IAM, networking, traffic, and security chapters. ACE leans heavily on them.
2. **Hands-on Qwiklabs / Skill Boost.** Pick paths that touch services you haven't personally used. ACE expects console familiarity.
3. **Week 2 — practice tests** from a reputable source (Google's official practice, ExamTopics, Tutorials Dojo). Don't memorise the questions; review the *explanations* for any you got wrong.
4. **Day-before review** — the keyword playbook above, and the distractor-traps list. The actual mechanics of how Spanner works don't matter on exam day; whether you can pick it over BigQuery does.
5. **Take the exam in a quiet, alert state.** ACE is not the hardest GCP exam, but it has a long tail of questions where the right answer requires both technical knowledge and careful reading.

## Where the course ends

Fourteen notebooks. Roughly six hours of audio. The same chapter spine as the AWS and Azure courses, with the GCP-unique pieces called out everywhere: global VPC, hierarchical IAM cascade, two-step WIF token dance, VPC Service Controls, multi-region storage as a first-class location type, sustained-use discounts, Spanner.

If the course did its job, you have a working mental model of GCP — not memorised facts, but a sense of what shape each service is and what trade-offs it represents. The blueprint coverage falls out of that mental model; so does most real-world architecture work.

Good luck on the exam, and good luck on whatever you build.